# Preparação para Modelagem

Este notebook consolida as decisões obtidas durante a análise exploratória e prepara a base enriquecida para o desenvolvimento dos modelos de classificação.

A estratégia experimental adota separação temporal entre os dados disponíveis:

- **2023:** conjunto de desenvolvimento, utilizado para preparação, seleção de variáveis, validação e escolha dos modelos;
- **2024:** conjunto de teste temporal final, mantido isolado durante as decisões de modelagem.

O objetivo é prever a condição de alfabetização dos alunos avaliados utilizando apenas informações consideradas disponíveis antes da avaliação, evitando vazamento de informação (*data leakage*).

As decisões adotadas neste notebook são fundamentadas nas análises realizadas no `01_eda_alunos_modelagem.ipynb`.

## 1. Objetivo e estratégia experimental

A modelagem será tratada como um problema de classificação binária, tendo `alfabetizado` como variável-alvo.

A população de modelagem é composta exclusivamente pelos alunos efetivamente avaliados, conforme definido na análise exploratória.

A estratégia adotada será:

- utilizar os dados de **2023** como conjunto de desenvolvimento;
- realizar seleção de variáveis, pré-processamento, treinamento, validação e escolha de modelos exclusivamente com os dados de 2023;
- utilizar `id_escola` para preservar o agrupamento dos alunos durante a validação;
- manter os dados de **2024 completamente isolados** até a avaliação temporal final;
- excluir identificadores, variáveis operacionais e características com evidência de vazamento de informação;
- utilizar informações externas do Censo Escolar respeitando a defasagem temporal:
  - avaliação de 2023 ← Censo Escolar 2022;
  - avaliação de 2024 ← Censo Escolar 2023.

Essa abordagem busca reproduzir uma situação mais próxima de uso real: estimar o risco de não alfabetização utilizando informações disponíveis antes da aplicação da avaliação.

## 2. Carregamento da base enriquecida

A base utilizada a partir deste ponto é o produto consolidado da etapa de enriquecimento realizada anteriormente.

Ela contém a população de alunos efetivamente avaliados e incorpora características contextuais do Censo Escolar agregadas por município e rede de ensino.

In [18]:
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/processed/alunos_modelagem_enriquecido.parquet")

df = pd.read_parquet(DATA_PATH)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")

Linhas: 3,354,661
Colunas: 97


## 3. Inventário das variáveis

Antes da definição do conjunto preditivo, as variáveis da base são inventariadas e classificadas de acordo com sua origem e função na modelagem.

Essa etapa permite separar explicitamente:

- variável-alvo;
- variáveis de controle e validação;
- identificadores;
- características provenientes da base Gold;
- metas de alfabetização;
- características contextuais provenientes do Censo Escolar.

As exclusões do conjunto preditivo serão realizadas posteriormente, com base nas evidências obtidas durante a análise exploratória.

In [19]:
inventario = pd.DataFrame({
    "posicao": range(1, len(df.columns) + 1),
    "variavel": df.columns,
    "tipo": [str(dtype) for dtype in df.dtypes],
})

inventario

,posicao,variavel,tipo
0,1,ano,Int64
1,2,id_municipio,string
2,3,id_municipio_nome,string
3,4,id_escola,string
4,5,id_aluno,string
...,...,...,...
92,93,cobertura_qt_doc_bas_pct,Float64
93,94,alunos_por_docente,Float64
94,95,alunos_por_sala,Float64
95,96,docentes_por_100_matriculas,Float64


### 3.1 Classificação funcional das variáveis

As variáveis são classificadas de acordo com sua função no experimento antes da definição do conjunto preditivo.

Uma variável não utilizada como *feature* pode continuar sendo necessária para controle temporal, validação, rastreabilidade ou avaliação do modelo.

A classificação considera as evidências obtidas durante a análise exploratória e separa as variáveis em:

- **alvo:** variável que será prevista;
- **controle:** utilizada na estratégia experimental, mas não como preditora;
- **identificador:** utilizada para rastreabilidade ou agrupamento;
- **exclusão por leakage/operacional:** informação indisponível ou inadequada no momento da previsão;
- **candidata Gold:** característica potencialmente utilizável proveniente da base original;
- **candidata Censo:** característica contextual construída a partir do Censo Escolar;
- **auditoria Censo:** característica mantida para controle de qualidade, mas não destinada ao conjunto preditivo.

In [20]:
def classificar_variavel(coluna):
    if coluna == "alfabetizado":
        return "alvo"

    if coluna in {"ano", "ano_censo"}:
        return "controle"

    if coluna in {
        "id_municipio",
        "id_municipio_nome",
        "id_escola",
        "id_aluno",
    }:
        return "identificador"

    if coluna in {
        "caderno",
        "presenca",
        "preenchimento_caderno",
        "proficiencia",
        "taxa_alfabetizacao",
        "media_portugues",
        "nivel_alfabetizacao",
        "percentual_participacao",
    }:
        return "exclusao_leakage_operacional"

    if coluna.startswith("cobertura_"):
        return "auditoria_censo"

    if coluna.startswith("meta_alfabetizacao_"):
        return "candidata_gold"

    if coluna in {"serie", "rede", "peso_aluno"}:
        return "candidata_gold"

    return "candidata_censo"


inventario["funcao"] = inventario["variavel"].apply(classificar_variavel)

inventario.groupby("funcao").size().sort_values(ascending=False)

funcao
candidata_censo                 43
auditoria_censo                 29
candidata_gold                  10
exclusao_leakage_operacional     8
identificador                    4
controle                         2
alvo                             1
dtype: int64

### 3.2 Revisão das variáveis candidatas da Gold

As variáveis classificadas inicialmente como `candidata_gold` exigem tratamento individual antes da definição do conjunto preditivo.

As decisões consideram as evidências já obtidas na análise exploratória:

- `serie` possui valor constante na população analisada e não acrescenta poder discriminativo;
- `rede` representa uma característica educacional disponível antes da avaliação e permanece como candidata;
- `peso_aluno` possui natureza amostral/estatística e não será utilizado diretamente como preditor;
- as metas de alfabetização apresentam forte redundância entre si;
- `meta_alfabetizacao_2030` é constante e não deve ser utilizada como feature;
- as demais metas devem ser reduzidas a uma representação não redundante, respeitando disponibilidade temporal.

In [21]:
candidatas_gold = inventario.loc[
    inventario["funcao"] == "candidata_gold",
    ["variavel", "tipo"]
].copy()

candidatas_gold

,variavel,tipo
6,serie,string
7,rede,str
12,peso_aluno,float64
17,meta_alfabetizacao_2024,float64
18,meta_alfabetizacao_2025,float64
19,meta_alfabetizacao_2026,float64
20,meta_alfabetizacao_2027,float64
21,meta_alfabetizacao_2028,float64
22,meta_alfabetizacao_2029,float64
23,meta_alfabetizacao_2030,float64


In [22]:
decisao_gold = {
    "serie": ("excluir", "variável constante na população de modelagem"),
    "rede": ("manter", "característica educacional disponível antes da avaliação"),
    "peso_aluno": ("excluir", "peso estatístico/amostral; não utilizado como preditor"),
    "meta_alfabetizacao_2024": ("excluir", "redundante com as demais metas"),
    "meta_alfabetizacao_2025": (
        "manter_provisoriamente",
        "representante das metas; disponibilidade temporal avaliada na EDA"
    ),
    "meta_alfabetizacao_2026": ("excluir", "redundante com meta_alfabetizacao_2025"),
    "meta_alfabetizacao_2027": ("excluir", "redundante com meta_alfabetizacao_2025"),
    "meta_alfabetizacao_2028": ("excluir", "redundante com meta_alfabetizacao_2025"),
    "meta_alfabetizacao_2029": ("excluir", "redundante com meta_alfabetizacao_2025"),
    "meta_alfabetizacao_2030": ("excluir", "variável constante"),
}

decisoes_gold = pd.DataFrame(
    [
        {
            "variavel": variavel,
            "decisao": decisao,
            "justificativa": justificativa,
        }
        for variavel, (decisao, justificativa) in decisao_gold.items()
    ]
)

decisoes_gold

,variavel,decisao,justificativa
0,serie,excluir,variável constante na população de modelagem
1,rede,manter,característica educacional disponível antes da...
2,peso_aluno,excluir,peso estatístico/amostral; não utilizado como ...
3,meta_alfabetizacao_2024,excluir,redundante com as demais metas
4,meta_alfabetizacao_2025,manter_provisoriamente,representante das metas; disponibilidade tempo...
5,meta_alfabetizacao_2026,excluir,redundante com meta_alfabetizacao_2025
6,meta_alfabetizacao_2027,excluir,redundante com meta_alfabetizacao_2025
7,meta_alfabetizacao_2028,excluir,redundante com meta_alfabetizacao_2025
8,meta_alfabetizacao_2029,excluir,redundante com meta_alfabetizacao_2025
9,meta_alfabetizacao_2030,excluir,variável constante


### 3.3 Revisão das características do Censo Escolar

As características provenientes do Censo Escolar representam informações contextuais agregadas por município e rede de ensino.

A análise exploratória identificou dois grupos distintos:

- características substantivas, relacionadas à infraestrutura, localização, matrículas, docentes e recursos educacionais;
- métricas `cobertura_*`, criadas durante o processamento para auditoria da completude dos dados.

As métricas de cobertura são preservadas na base para rastreabilidade e controle de qualidade, mas não serão utilizadas como preditores. A EDA mostrou que essas variáveis apresentam baixa variabilidade e forte redundância entre si.

A seleção das características substantivas será realizada exclusivamente sobre o conjunto de desenvolvimento de 2023. O conjunto temporal de 2024 não participa de nenhuma decisão de seleção de variáveis.

In [23]:
candidatas_censo = inventario.loc[
    inventario["funcao"] == "candidata_censo",
    "variavel"
].tolist()

print(f"Features substantivas candidatas: {len(candidatas_censo)}")

pd.DataFrame({
    "variavel": candidatas_censo
})

Features substantivas candidatas: 43


,variavel
0,TP_DEPENDENCIA
1,qtd_escolas
2,pct_info_detalhada_completa
3,pct_tp_localizacao_1
4,pct_tp_localizacao_2
5,pct_localizacao_diferenciada
6,pct_in_agua_potavel
7,pct_in_agua_rede_publica
8,pct_in_energia_rede_publica
9,pct_in_esgoto_rede_publica


In [24]:
df_dev = df.loc[df["ano"] == 2023]

variabilidade_censo = pd.DataFrame({
    "variavel": candidatas_censo,
    "n_unicos": [
        df_dev[coluna].nunique(dropna=True)
        for coluna in candidatas_censo
    ],
    "pct_nulos": [
        df_dev[coluna].isna().mean() * 100
        for coluna in candidatas_censo
    ],
})

variabilidade_censo = variabilidade_censo.sort_values(
    ["n_unicos", "variavel"]
).reset_index(drop=True)

variabilidade_censo

,variavel,n_unicos,pct_nulos
0,TP_DEPENDENCIA,2,0.000000
1,qtd_escolas,172,0.000000
2,pct_in_energia_rede_publica,211,0.000000
3,pct_in_acessibilidade_sinal_visual,326,0.000000
4,pct_in_laboratorio_ciencias,330,0.000000
5,total_qt_tablet_aluno,331,0.000000
6,pct_in_agua_potavel,336,0.000000
7,total_qt_comp_portatil_aluno,345,0.000000
8,pct_localizacao_diferenciada,401,0.000000
9,total_qt_desktop_aluno,432,0.000000


#### 3.3.1 Redundância entre características substantivas

Nenhuma das 43 características substantivas do Censo Escolar apresentou ausência de variabilidade no conjunto de desenvolvimento de 2023.

A análise de redundância é realizada exclusivamente sobre 2023 e considera as características numéricas. Pares com correlação absoluta igual ou superior a 0,95 são identificados para revisão.

A correlação elevada não implica exclusão automática. A decisão também considera a semântica das variáveis e se ambas representam essencialmente a mesma informação.

In [25]:
censo_numericas = [
    coluna
    for coluna in candidatas_censo
    if pd.api.types.is_numeric_dtype(df_dev[coluna])
]

corr_censo = (
    df_dev[censo_numericas]
    .corr()
    .abs()
)

pares_redundantes = []

for i, coluna_a in enumerate(censo_numericas):
    for coluna_b in censo_numericas[i + 1:]:
        correlacao = corr_censo.loc[coluna_a, coluna_b]

        if correlacao >= 0.95:
            pares_redundantes.append({
                "variavel_a": coluna_a,
                "variavel_b": coluna_b,
                "correlacao_abs": correlacao,
            })

pares_redundantes = (
    pd.DataFrame(pares_redundantes)
    .sort_values("correlacao_abs", ascending=False)
    .reset_index(drop=True)
)

print(f"Pares com |correlação| >= 0,95: {len(pares_redundantes)}")

pares_redundantes

Pares com |correlação| >= 0,95: 8


,variavel_a,variavel_b,correlacao_abs
0,pct_tp_localizacao_1,pct_tp_localizacao_2,1.000000
1,total_qt_salas_utilizadas,total_qt_mat_bas,0.994815
2,total_qt_mat_bas,total_qt_doc_bas,0.990558
3,total_qt_salas_utilizadas,total_qt_doc_bas,0.990335
4,pct_info_detalhada_completa,pct_in_internet,0.989601
5,qtd_escolas,total_qt_salas_utilizadas,0.989494
6,qtd_escolas,total_qt_mat_bas,0.983129
7,qtd_escolas,total_qt_doc_bas,0.971170


In [26]:
variaveis_redundancia = [
    "qtd_escolas",
    "total_qt_salas_utilizadas",
    "total_qt_mat_bas",
    "total_qt_doc_bas",
    "alunos_por_docente",
    "alunos_por_sala",
    "docentes_por_100_matriculas",
    "pct_info_detalhada_completa",
    "pct_in_internet",
]

df_dev[variaveis_redundancia].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
]

,mean,std,min,25%,50%,75%,max
qtd_escolas,123.805363,271.703455,1.0,17.0,40.0,106.0,1564.0
total_qt_salas_utilizadas,1231.162076,2941.568424,2.0,120.0,314.0,918.0,16710.0
total_qt_mat_bas,46933.21825,111967.928668,94.0,3647.0,10484.0,34615.0,625433.0
total_qt_doc_bas,2325.3909,5001.029697,8.0,223.0,594.0,1900.0,27229.0
alunos_por_docente,17.940131,4.399024,4.424242,14.682243,17.519231,21.188197,43.803625
alunos_por_sala,34.589176,8.896226,4.717391,28.176471,34.910891,40.52802,175.0
docentes_por_100_matriculas,5.940693,1.580848,2.282916,4.719609,5.708013,6.810948,22.60274
pct_info_detalhada_completa,89.874506,17.394136,0.0,88.0,97.69821,100.0,100.0
pct_in_internet,90.665614,17.457792,0.0,89.380531,99.528302,100.0,100.0


In [27]:
exclusoes_censo = {
    "pct_tp_localizacao_2": (
        "complemento exato de pct_tp_localizacao_1"
    ),
    "total_qt_salas_utilizadas": (
        "redundante com o indicador de porte da rede"
    ),
    "total_qt_mat_bas": (
        "redundante com o indicador de porte da rede"
    ),
    "total_qt_doc_bas": (
        "redundante com o indicador de porte da rede"
    ),
    "docentes_por_100_matriculas": (
        "transformação inversamente relacionada a alunos_por_docente"
    ),
    "pct_info_detalhada_completa": (
        "indicador de completude da fonte e altamente redundante com pct_in_internet"
    ),
    "TP_DEPENDENCIA": (
    "campo técnico do Censo redundante com a variável rede"
    ),
    "total_qt_tablet_aluno": (
    "redundante com media_qt_tablet_aluno e influenciado pelo porte da rede"
    ),
}

features_censo_selecionadas = [
    coluna
    for coluna in candidatas_censo
    if coluna not in exclusoes_censo
]

print(f"Candidatas originais: {len(candidatas_censo)}")
print(f"Excluídas: {len(exclusoes_censo)}")
print(f"Selecionadas: {len(features_censo_selecionadas)}")

Candidatas originais: 43
Excluídas: 8
Selecionadas: 35


#### 3.3.2 Validação da redundância após seleção

Após as exclusões definidas por redundância estatística e semântica, a correlação entre as características substantivas selecionadas é recalculada sobre o conjunto de desenvolvimento de 2023.

O objetivo é verificar se permanecem pares com correlação absoluta igual ou superior a 0,95.

In [28]:
censo_numericas_selecionadas = [
    coluna
    for coluna in features_censo_selecionadas
    if pd.api.types.is_numeric_dtype(df_dev[coluna])
]

corr_censo_selecionado = (
    df_dev[censo_numericas_selecionadas]
    .corr(method="spearman")
    .abs()
)

pares_redundantes_pos = []

for i, coluna_a in enumerate(censo_numericas_selecionadas):
    for coluna_b in censo_numericas_selecionadas[i + 1:]:
        correlacao = corr_censo_selecionado.loc[coluna_a, coluna_b]

        if correlacao >= 0.95:
            pares_redundantes_pos.append({
                "variavel_a": coluna_a,
                "variavel_b": coluna_b,
                "correlacao_abs": correlacao,
            })

pares_redundantes_pos = pd.DataFrame(pares_redundantes_pos)

print(
    "Pares restantes com |correlação| >= 0,95:",
    len(pares_redundantes_pos)
)

pares_redundantes_pos

Pares restantes com |correlação| >= 0,95: 0


""


### 3.4 Conjunto de características para modelagem

Após a revisão das variáveis da Gold e das características contextuais do Censo Escolar, é consolidado o conjunto de preditores que seguirá para a etapa de modelagem.

As decisões adotadas foram:

- exclusão de identificadores do conjunto preditivo;
- exclusão de variáveis operacionais e com evidência de *data leakage*;
- exclusão das métricas `cobertura_*`, mantidas apenas para auditoria;
- exclusão de características constantes ou estruturalmente redundantes;
- preservação de `rede` como característica categórica;
- utilização provisória de `meta_alfabetizacao_2025` como representante das metas de alfabetização;
- seleção de 37 características substantivas do Censo Escolar.

As decisões de seleção foram realizadas exclusivamente sobre o conjunto de desenvolvimento de 2023. O conjunto temporal de 2024 permanece isolado.

In [29]:
features_gold_selecionadas = [
    "rede",
    "meta_alfabetizacao_2025",
]

features_modelagem = (
    features_gold_selecionadas
    + features_censo_selecionadas
)

print(f"Features Gold: {len(features_gold_selecionadas)}")
print(f"Features Censo: {len(features_censo_selecionadas)}")
print(f"Total de features: {len(features_modelagem)}")

pd.DataFrame({
    "variavel": features_modelagem,
    "origem": (
        ["Gold"] * len(features_gold_selecionadas)
        + ["Censo Escolar"] * len(features_censo_selecionadas)
    )
})

Features Gold: 2
Features Censo: 35
Total de features: 37


,variavel,origem
0,rede,Gold
1,meta_alfabetizacao_2025,Gold
2,qtd_escolas,Censo Escolar
3,pct_tp_localizacao_1,Censo Escolar
4,pct_localizacao_diferenciada,Censo Escolar
5,pct_in_agua_potavel,Censo Escolar
6,pct_in_agua_rede_publica,Censo Escolar
7,pct_in_energia_rede_publica,Censo Escolar
8,pct_in_esgoto_rede_publica,Censo Escolar
9,pct_in_lixo_servico_coleta,Censo Escolar


In [30]:
assert len(features_modelagem) == len(set(features_modelagem))
assert set(features_modelagem).issubset(df.columns)

print("Conjunto de features validado.")

Conjunto de features validado.


## 4. Separação temporal e definição do target

A estratégia experimental utiliza uma separação temporal explícita:

- **2023:** conjunto de desenvolvimento, utilizado para treinamento, validação, seleção de modelos e ajuste de hiperparâmetros;
- **2024:** conjunto de teste temporal final, mantido isolado até a avaliação do modelo selecionado.

O problema é formulado como classificação de risco. Por isso, a variável-alvo é codificada como:

- `1` — aluno **não alfabetizado**;
- `0` — aluno **alfabetizado**.

Essa convenção torna métricas como *recall*, *precision* e F1 diretamente interpretáveis em relação à identificação dos alunos em risco de não alfabetização.

O identificador `id_escola` não é utilizado como preditor. Ele é preservado separadamente para garantir que alunos da mesma escola permaneçam no mesmo grupo durante a validação do conjunto de desenvolvimento.

In [31]:
TARGET = "alfabetizado"
GROUP = "id_escola"

mask_dev = df["ano"] == 2023
mask_test = df["ano"] == 2024

X_dev = df.loc[mask_dev, features_modelagem].copy()
X_test = df.loc[mask_test, features_modelagem].copy()

y_dev = (
    df.loc[mask_dev, TARGET]
    .map({"Sim": 0, "Não": 1})
    .astype("int8")
)

y_test = (
    df.loc[mask_test, TARGET]
    .map({"Sim": 0, "Não": 1})
    .astype("int8")
)

groups_dev = df.loc[mask_dev, GROUP].copy()

print(f"Desenvolvimento 2023: {len(X_dev):,}")
print(f"Teste temporal 2024: {len(X_test):,}")
print(f"Features: {X_dev.shape[1]}")
print(f"Escolas no desenvolvimento: {groups_dev.nunique():,}")

Desenvolvimento 2023: 1,502,809
Teste temporal 2024: 1,851,852
Features: 37
Escolas no desenvolvimento: 36,525


In [32]:
distribuicao_target = pd.DataFrame({
    "quantidade": y_dev.value_counts().sort_index(),
    "percentual": y_dev.value_counts(normalize=True).sort_index() * 100,
})

distribuicao_target.index = [
    "0 - Alfabetizado",
    "1 - Não alfabetizado",
]

distribuicao_target

,quantidade,percentual
0 - Alfabetizado,877427,58.385796
1 - Não alfabetizado,625382,41.614204


## 5. Estratégia de validação cruzada

A avaliação interna do conjunto de desenvolvimento de 2023 utiliza validação cruzada estratificada e agrupada por escola.

O `StratifiedGroupKFold` foi escolhido para atender simultaneamente a dois requisitos:

- impedir que alunos da mesma escola apareçam nos conjuntos de treino e validação de um mesmo fold;
- buscar a preservação da distribuição da variável-alvo entre os folds.

Essa estratégia reduz o risco de resultados excessivamente otimistas decorrentes do compartilhamento do contexto escolar entre treino e validação.

O conjunto temporal de 2024 permanece completamente fora desse processo.

In [33]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
RANDOM_STATE = 42

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

resumo_folds = []

for fold, (idx_treino, idx_validacao) in enumerate(
    cv.split(X_dev, y_dev, groups=groups_dev),
    start=1,
):
    y_treino_fold = y_dev.iloc[idx_treino]
    y_validacao_fold = y_dev.iloc[idx_validacao]

    escolas_treino = set(groups_dev.iloc[idx_treino])
    escolas_validacao = set(groups_dev.iloc[idx_validacao])

    resumo_folds.append({
        "fold": fold,
        "n_treino": len(idx_treino),
        "n_validacao": len(idx_validacao),
        "risco_treino_pct": y_treino_fold.mean() * 100,
        "risco_validacao_pct": y_validacao_fold.mean() * 100,
        "escolas_treino": len(escolas_treino),
        "escolas_validacao": len(escolas_validacao),
        "escolas_em_comum": len(
            escolas_treino.intersection(escolas_validacao)
        ),
    })

resumo_folds = pd.DataFrame(resumo_folds)

resumo_folds

,fold,n_treino,n_validacao,risco_treino_pct,risco_validacao_pct,escolas_treino,escolas_validacao,escolas_em_comum
0,1,1202258,300551,41.614362,41.613570,29231,7294,0
1,2,1202246,300563,41.613863,41.615568,29225,7300,0
2,3,1202238,300571,41.614472,41.613130,29230,7295,0
3,4,1202224,300585,41.614125,41.614518,29194,7331,0
4,5,1202270,300539,41.614196,41.614233,29220,7305,0


In [34]:
assert (resumo_folds["escolas_em_comum"] == 0).all()

print(
    "Variação da classe de risco na validação:",
    f"{resumo_folds['risco_validacao_pct'].min():.2f}% - "
    f"{resumo_folds['risco_validacao_pct'].max():.2f}%"
)

print("Separação por escola validada.")

Variação da classe de risco na validação: 41.61% - 41.62%
Separação por escola validada.


## 6. Preparação do pipeline de pré-processamento

O pré-processamento será incorporado ao pipeline de modelagem para evitar vazamento de informação entre os folds da validação cruzada.

As transformações serão ajustadas exclusivamente sobre os dados de treinamento de cada fold e posteriormente aplicadas à respectiva validação.

Nesta etapa, as características são separadas entre numéricas e categóricas antes da definição das transformações.

In [35]:
features_categoricas = [
    coluna
    for coluna in features_modelagem
    if not pd.api.types.is_numeric_dtype(X_dev[coluna])
]

features_numericas = [
    coluna
    for coluna in features_modelagem
    if pd.api.types.is_numeric_dtype(X_dev[coluna])
]

print(f"Features categóricas: {len(features_categoricas)}")
print(features_categoricas)

print(f"\nFeatures numéricas: {len(features_numericas)}")
print(features_numericas)

Features categóricas: 1
['rede']

Features numéricas: 36
['meta_alfabetizacao_2025', 'qtd_escolas', 'pct_tp_localizacao_1', 'pct_localizacao_diferenciada', 'pct_in_agua_potavel', 'pct_in_agua_rede_publica', 'pct_in_energia_rede_publica', 'pct_in_esgoto_rede_publica', 'pct_in_lixo_servico_coleta', 'pct_in_banheiro_pne', 'pct_in_biblioteca', 'pct_in_biblioteca_sala_leitura', 'pct_in_laboratorio_ciencias', 'pct_in_laboratorio_informatica', 'pct_in_patio_coberto', 'pct_in_patio_descoberto', 'pct_in_parque_infantil', 'pct_in_quadra_esportes', 'pct_in_acessibilidade_corrimao', 'pct_in_acessibilidade_rampas', 'pct_in_acessibilidade_sinal_visual', 'pct_in_computador', 'pct_in_internet', 'pct_in_internet_aprendizagem', 'pct_in_banda_larga', 'media_qt_salas_utilizadas', 'media_qt_desktop_aluno', 'total_qt_desktop_aluno', 'media_qt_comp_portatil_aluno', 'total_qt_comp_portatil_aluno', 'media_qt_tablet_aluno', 'media_qt_mat_bas', 'media_qt_doc_bas', 'alunos_por_docente', 'alunos_por_sala', 'dispos

### 6.1 Pré-processamento para modelos lineares

Para os modelos lineares, o pré-processamento é composto por:

- imputação das variáveis numéricas pela mediana;
- padronização das variáveis numéricas com `StandardScaler`;
- imputação da variável categórica pela categoria mais frequente;
- codificação da variável `rede` por `OneHotEncoder`.

Todas as transformações permanecem dentro do pipeline do scikit-learn. Dessa forma, seus parâmetros são aprendidos somente sobre os dados de treinamento de cada fold, evitando vazamento de informação durante a validação cruzada.

In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pipeline_numerico_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
            ),
        ),
    ]
)

preprocessor_linear = ColumnTransformer(
    transformers=[
        (
            "numeric",
            pipeline_numerico_linear,
            features_numericas,
        ),
        (
            "categorical",
            pipeline_categorico,
            features_categoricas,
        ),
    ],
    remainder="drop",
)

print(preprocessor_linear)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['meta_alfabetizacao_2025', 'qtd_escolas',
                                  'pct_tp_localizacao_1',
                                  'pct_localizacao_diferenciada',
                                  'pct_in_agua_potavel',
                                  'pct_in_agua_rede_publica',
                                  'pct_in_energia_rede_publica',
                                  'pct_in_esgoto_rede_publica',
                                  'pct_in_lixo_...
                                  'pct_in_internet_aprendizagem',
                                  'pct_in_banda_larga',
                                  'media_qt_salas_utilizadas',
                                  'media_qt_d

## 7. Modelagem

A modelagem é iniciada com um baseline simples, utilizado como referência para avaliar se os modelos supervisionados efetivamente acrescentam capacidade preditiva.

Todo o processo de avaliação utiliza exclusivamente o conjunto de desenvolvimento de 2023 e a estratégia de validação cruzada definida anteriormente. O conjunto temporal de 2024 permanece reservado para a avaliação final.

In [37]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate

baseline = DummyClassifier(
    strategy="prior",
)

metricas = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

resultado_baseline = cross_validate(
    baseline,
    X_dev,
    y_dev,
    groups=groups_dev,
    cv=cv,
    scoring=metricas,
    n_jobs=-1,
)

resumo_baseline = pd.DataFrame({
    metrica: [
        resultado_baseline[f"test_{metrica}"].mean(),
        resultado_baseline[f"test_{metrica}"].std(),
    ]
    for metrica in metricas
}, index=["media", "desvio_padrao"])

resumo_baseline.T

/home/erick/Projetos/tech_challenge_3_alfabetizacao/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/erick/Projetos/tech_challenge_3_alfabetizacao/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/erick/Projetos/tech_challenge_3_alfabetizacao/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavio

,media,desvio_padrao
roc_auc,0.500000,0.000000
average_precision,0.416142,0.000008
balanced_accuracy,0.500000,0.000000
precision,0.000000,0.000000
recall,0.000000,0.000000
f1,0.000000,0.000000


#### Resultado do baseline

O `DummyClassifier` apresentou ROC-AUC e balanced accuracy iguais a 0,50, comportamento esperado para um classificador sem capacidade discriminativa.

Como a classe majoritária do conjunto de desenvolvimento corresponde aos alunos alfabetizados, o baseline não identificou alunos da classe positiva de risco (`Não alfabetizado`), resultando em precision, recall e F1 iguais a zero.

Esses resultados estabelecem a referência mínima para comparação com os modelos supervisionados.

### 7.2 Regressão Logística

A Regressão Logística é utilizada como primeiro modelo supervisionado por oferecer uma referência linear interpretável para comparação com modelos mais complexos.

O pré-processamento é incorporado ao pipeline para garantir que imputação, padronização e codificação sejam ajustadas exclusivamente sobre o conjunto de treinamento de cada fold.

In [38]:
from sklearn.linear_model import LogisticRegression

modelo_logistico = Pipeline(
    steps=[
        ("preprocessor", preprocessor_linear),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

resultado_logistico = cross_validate(
    modelo_logistico,
    X_dev,
    y_dev,
    groups=groups_dev,
    cv=cv,
    scoring=metricas,
    n_jobs=-1,
)

resumo_logistico = pd.DataFrame({
    metrica: [
        resultado_logistico[f"test_{metrica}"].mean(),
        resultado_logistico[f"test_{metrica}"].std(),
    ]
    for metrica in metricas
}, index=["media", "desvio_padrao"])

resumo_logistico.T

,media,desvio_padrao
roc_auc,0.680330,0.001840
average_precision,0.588163,0.001541
balanced_accuracy,0.605193,0.000767
precision,0.611163,0.002431
recall,0.384987,0.004238
f1,0.472376,0.002692


In [39]:
comparacao_inicial = pd.DataFrame({
    "baseline": resumo_baseline.loc["media"],
    "regressao_logistica": resumo_logistico.loc["media"],
})

comparacao_inicial

,baseline,regressao_logistica
roc_auc,0.500000,0.680330
average_precision,0.416142,0.588163
balanced_accuracy,0.500000,0.605193
precision,0.000000,0.611163
recall,0.000000,0.384987
f1,0.000000,0.472376


#### Resultado da Regressão Logística

A Regressão Logística apresentou desempenho superior ao baseline em todas as métricas avaliadas, alcançando ROC-AUC médio de 0,680 e Average Precision de 0,588.

Os resultados apresentaram baixa variação entre os cinco folds, indicando estabilidade da capacidade preditiva entre diferentes grupos de escolas.

Considerando a classe positiva como `Não alfabetizado`, o modelo apresentou precision de 0,611 e recall de 0,385 com o limiar padrão de classificação. Portanto, embora exista capacidade discriminativa acima do baseline, uma parcela relevante dos alunos em risco ainda não é identificada.

Nenhum ajuste de limiar ou hiperparâmetro é realizado nesta etapa, pois essas decisões serão avaliadas após a comparação entre os modelos candidatos.

### 7.3 HistGradientBoosting

O `HistGradientBoostingClassifier` é utilizado como primeiro modelo não linear baseado em árvores.

Esse algoritmo permite capturar relações não lineares e interações entre as características, servindo como contraponto à Regressão Logística.

Como modelos baseados em árvores não dependem da escala das variáveis, o pré-processamento utilizado nesta etapa não aplica padronização às características numéricas.

In [40]:
pipeline_numerico_arvore = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocessor_arvore = ColumnTransformer(
    transformers=[
        (
            "numeric",
            pipeline_numerico_arvore,
            features_numericas,
        ),
        (
            "categorical",
            pipeline_categorico,
            features_categoricas,
        ),
    ],
    remainder="drop",
)

In [41]:
from sklearn.ensemble import HistGradientBoostingClassifier

modelo_hist_gradient = Pipeline(
    steps=[
        ("preprocessor", preprocessor_arvore),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

resultado_hist_gradient = cross_validate(
    modelo_hist_gradient,
    X_dev,
    y_dev,
    groups=groups_dev,
    cv=cv,
    scoring=metricas,
    n_jobs=-1,
)

resumo_hist_gradient = pd.DataFrame({
    metrica: [
        resultado_hist_gradient[f"test_{metrica}"].mean(),
        resultado_hist_gradient[f"test_{metrica}"].std(),
    ]
    for metrica in metricas
}, index=["media", "desvio_padrao"])

resumo_hist_gradient.T

,media,desvio_padrao
roc_auc,0.683782,0.001357
average_precision,0.590036,0.001112
balanced_accuracy,0.613098,0.001692
precision,0.602745,0.002130
recall,0.426637,0.007958
f1,0.499572,0.005065


In [42]:
comparacao_modelos = pd.DataFrame({
    "baseline": resumo_baseline.loc["media"],
    "regressao_logistica": resumo_logistico.loc["media"],
    "hist_gradient_boosting": resumo_hist_gradient.loc["media"],
})

comparacao_modelos

,baseline,regressao_logistica,hist_gradient_boosting
roc_auc,0.500000,0.680330,0.683782
average_precision,0.416142,0.588163,0.590036
balanced_accuracy,0.500000,0.605193,0.613098
precision,0.000000,0.611163,0.602745
recall,0.000000,0.384987,0.426637
f1,0.000000,0.472376,0.499572


#### Resultado do HistGradientBoosting

O HistGradientBoosting apresentou desempenho superior à Regressão Logística na maior parte das métricas avaliadas.

O modelo alcançou ROC-AUC médio de 0,684, Average Precision de 0,590 e balanced accuracy de 0,613. Para a identificação dos alunos em risco, o recall aumentou de 0,385 na Regressão Logística para 0,427, enquanto o F1 aumentou de 0,472 para 0,500.

A melhora ocorreu com pequena redução da precision, de 0,611 para 0,603. Os resultados permaneceram estáveis entre os folds.

Os resultados indicam ganho com a utilização de relações não lineares, embora a diferença em relação ao modelo linear seja moderada. A comparação ainda utiliza os hiperparâmetros padrão e o limiar padrão de classificação.

## 8. Otimização do modelo

Após a comparação inicial, o HistGradientBoosting apresentou o melhor desempenho entre os modelos avaliados com hiperparâmetros padrão.

A otimização é realizada exclusivamente sobre o conjunto de desenvolvimento de 2023, preservando o conjunto temporal de 2024 para a avaliação final.

A busca é mantida deliberadamente restrita aos principais hiperparâmetros relacionados à complexidade e regularização do modelo, evitando uma exploração excessiva do espaço de parâmetros.

In [44]:
from sklearn.model_selection import RandomizedSearchCV

parametros_hist = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [100, 200],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__l2_regularization": [0.0, 1.0, 5.0],
}

busca_hist = RandomizedSearchCV(
    estimator=modelo_hist_gradient,
    param_distributions=parametros_hist,
    n_iter=10,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

busca_hist.fit(
    X_dev,
    y_dev,
    groups=groups_dev,
)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/home/erick/Projetos/tech_challenge_3_alfabetizacao/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__l2_regularization': [0.0, 1.0, ...], 'model__learning_rate': [0.05, 0.1], 'model__max_iter': [100, 200], 'model__max_leaf_nodes': [15, 31, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedGro... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing 

In [45]:
print("Melhor Average Precision:")
print(f"{busca_hist.best_score_:.6f}")

print("\nMelhores hiperparâmetros:")
busca_hist.best_params_

Melhor Average Precision:
0.592231

Melhores hiperparâmetros:


{'model__max_leaf_nodes': 31,
 'model__max_iter': 100,
 'model__learning_rate': 0.05,
 'model__l2_regularization': 1.0}

#### Resultado da otimização

A busca de hiperparâmetros obteve Average Precision médio de 0,5922, frente a 0,5900 do HistGradientBoosting com configuração padrão.

A melhor configuração encontrada foi:

- `learning_rate = 0.05`;
- `max_iter = 100`;
- `max_leaf_nodes = 31`;
- `l2_regularization = 1.0`.

O ganho obtido foi pequeno, indicando que o desempenho do modelo é relativamente estável em relação às configurações avaliadas. A configuração otimizada é mantida como candidata para as etapas seguintes.

### 8.2 Avaliação do modelo otimizado

A melhor configuração encontrada na busca de hiperparâmetros é reavaliada utilizando as mesmas métricas e a mesma estratégia de validação cruzada empregadas na comparação inicial.

Isso permite verificar se o ganho observado em Average Precision também se reflete nas demais métricas relevantes para o problema.

In [46]:
modelo_hist_otimizado = busca_hist.best_estimator_

resultado_hist_otimizado = cross_validate(
    modelo_hist_otimizado,
    X_dev,
    y_dev,
    groups=groups_dev,
    cv=cv,
    scoring=metricas,
    n_jobs=-1,
)

resumo_hist_otimizado = pd.DataFrame({
    metrica: [
        resultado_hist_otimizado[f"test_{metrica}"].mean(),
        resultado_hist_otimizado[f"test_{metrica}"].std(),
    ]
    for metrica in metricas
}, index=["media", "desvio_padrao"])

resumo_hist_otimizado.T

,media,desvio_padrao
roc_auc,0.685198,0.001308
average_precision,0.592231,0.001000
balanced_accuracy,0.613648,0.001833
precision,0.604995,0.002464
recall,0.425190,0.008015
f1,0.499346,0.005171


In [47]:
comparacao_otimizacao = pd.DataFrame({
    "regressao_logistica": resumo_logistico.loc["media"],
    "hist_padrao": resumo_hist_gradient.loc["media"],
    "hist_otimizado": resumo_hist_otimizado.loc["media"],
})

comparacao_otimizacao

,regressao_logistica,hist_padrao,hist_otimizado
roc_auc,0.680330,0.683782,0.685198
average_precision,0.588163,0.590036,0.592231
balanced_accuracy,0.605193,0.613098,0.613648
precision,0.611163,0.602745,0.604995
recall,0.384987,0.426637,0.425190
f1,0.472376,0.499572,0.499346


#### Avaliação da configuração otimizada

A configuração otimizada apresentou ganho discreto em relação ao HistGradientBoosting padrão.

O ROC-AUC aumentou de 0,6838 para 0,6852 e a Average Precision, utilizada como métrica de otimização, aumentou de 0,5900 para 0,5922.

As métricas dependentes do limiar padrão permaneceram praticamente estáveis: o recall passou de 0,4266 para 0,4252 e o F1 de 0,4996 para 0,4993.

Assim, a otimização produziu uma pequena melhora na capacidade de ordenação dos alunos por risco, sem alteração relevante no desempenho de classificação com limiar de 0,5. A configuração otimizada é mantida como modelo candidato para as etapas seguintes.